<a href="https://colab.research.google.com/github/LukeRDuob/AI-Labsheets/blob/main/Lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction
This worksheet covers the **Self-Attention algorithm** we looked at in ArchitectureⅡ. Similar to last week, you will do some work implementing your own versions of these algorithms, to ensure that you understand the details of them.

# Step by step coding
We will code Self-Attention in PyTorch step by step.

#### 1. Import key packages: torch, and any others that you prefer to work with. In general, when writing code, you will put all your import statements at the top. However, for these worksheets we will import as we go along.

In [33]:
# TODO: import the torch package here
import torch as t

#### 2. Set the number of embedding values per token, in Architecture Ⅱ.

In [34]:
model_dim = 2

#### 3. Initialize the Linear Layers that we will use to create the Query, Key and Value. W_q, W_k and W_v represents the weights of Query, Key and Value, seperately.

In [43]:
W_q_linear = t.nn.Linear(model_dim, model_dim)
W_k_linear = t.nn.Linear(model_dim, model_dim)
W_v_linear = t.nn.Linear(model_dim, model_dim)


#### 4. Create a matrix of encoded_values as the input of self-attention module


In [44]:
token_len = 2
encoded_values = t.rand((token_len,model_dim))

#### 5. Create the query, key and values using the encoded values through corresponding linear layers.

In [45]:
q = W_q_linear(encoded_values)
k = W_k_linear(encoded_values)
v = W_v_linear(encoded_values)

#### 6. Multiply the Query matrix by the transpose of the Key matrix to compute the similarities scores.


In [46]:
# using torch.matmul function
scores = t.matmul(q,k.T)


#### 7. Scale the matrix of dot product similarities.

In [49]:
# the scaling dimension is k.shape[-1]
scores = scores/((k.shape[-1])**0.5)
print(scores)

tensor([[0.2428, 0.4643],
        [0.2738, 0.4333]], grad_fn=<DivBackward0>)


#### 8. Take the SoftMax of each row in the matrix of scaled Dot Product similarities.


In [50]:
# use torch.nn.functional.softmax
scores = t.nn.functional.softmax(scores, dim=-1)

#### 9. Multiply by the Values in matrix V to scale the values by their associated percentages and add them up.

In [51]:
# using torch.matmul function
attn = t.matmul(scores,v)
# you can also print out the attn output
print(attn)

tensor([[ 0.2096, -0.9416],
        [ 0.2034, -0.9507]], grad_fn=<MmBackward0>)


# Wrapped inside a self-attention class
We can organize and encapsulate the code above into a self-attention class with both initialization and forward computation.

In [52]:
class SelfAttention(t.nn.Module):

    def __init__(self, model_dim=2):
        super().__init__()
        # Initialize the Linear Layers that we will use to create the Query, Key and Value.
        # W_q, W_k and W_v represents the weights of Query, Key and Value, seperately.
        self.W_q_linear = t.nn.Linear(model_dim, model_dim)
        self.W_k_linear = t.nn.Linear(model_dim, model_dim)
        self.W_v_linear = t.nn.Linear(model_dim, model_dim)

    def forward(self, encoded_values):
        # Create the query, key and values using the encoded values through corresponding linear layers.
        q = W_q_linear(encoded_values)
        k = W_k_linear(encoded_values)
        v = W_v_linear(encoded_values)

        # Multiply the Query matrix by the transpose of the Key matrix to compute the similarities scores.
        scores = t.matmul(q,k.T)


        ## Scale the matrix of dot product similarities.
        scores = scores/((k.shape[-1])**0.5)


        ## Take the SoftMax of each row in the matrix of scaled Dot Product similarities
        scores = t.nn.functional.softmax(scores, dim=-1)

        ## Multiply by the Values in matrix V to scale the values by their associated percentages and add them up.
        attn = t.matmul(scores,v)

        return attn

Use this Self-Attention Class

In [53]:
# set the token_len and create a random encoded_values
token_len= 2
encoded_values = t.rand((token_len,model_dim))
# create a self-attention ojbect
selfAttention = SelfAttention()
# calculate attention for the token encodings
attn = selfAttention.forward(encoded_values=encoded_values)
# print out the attn output
print(attn)

tensor([[ 0.2176, -0.8906],
        [ 0.2183, -0.8865]], grad_fn=<MmBackward0>)
